# FK09 Vehicle Dynamics, Lap Simulation and 2026 Points Tool

Everything here follows the Zacharelis thesis. **Change the numbers in part 2, then use Run > Run All Cells.**

| Part | What it does |
|---|---|
| 1 | Setup |
| 2 | **Car parameters (the only cell you must edit)** |
| 3 | Check the inputs: tire, engine, gears |
| 4 | Single scenarios: acceleration, braking, skidpad, GGV diagram |
| 5 | Tracks (autocross, endurance) and how to import your own |
| 6 | Lap simulation: speed, channels, KPIs |
| 7 | **2026 points, goals, what-if (the design tool)** |
| 8 | Aero map: ride height, roll, yaw effects |
| 9 | Suspension: rates, quarter car, dampers, 7-post rig, sweeps |
| 10 | Yaw moment diagram |
| 11 | Correlation: match the sim to the real car |

The default car is the **thesis baseline car**. Everything marked PLACEHOLDER in `lapsim.py` is made up until we enter FK09 data.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
import lapsim
importlib.reload(lapsim)   # pick up edits to lapsim.py without restarting the kernel

pd.set_option("display.width", 200)
tracks = lapsim.load_default_tracks()   # autocross + endurance, traced from the competition maps (approximate)
results = lapsim.load_results()         # 2026 Michigan results
fastest_times = lapsim.field_best_times(results)

## 2. Car parameters (settings.json)
**Every setting lives in `settings.json`; edit it there, save, and re-run this cell** (then the cells below). SI units (kg, m, N). Sections: `tire`, `suspension`, `brakes`, `correlation`, `effects`, `car`. Every parameter is listed with a PLACEHOLDER value (thesis baseline car NTUA P19, plus made-up tire, engine and aero data), so Run All works as is. Replace the numbers with FK09 data as you get them; delete a line to fall back to the default in `lapsim.py`. A misspelled name is an error.
- `car.aero_mode`: `"constant"`, `"sensitivity"` or `"map"` (roll/yaw sensitivity or the aero map, part 8). `car.aero_map`: `"placeholder"`, the path of a CSV (columns frh, rrh, roll, yaw, czt, ab, cx) or `null`.
- **`effects`** switches physical effects and simplifying assumptions of the thesis on (`true`) or off (`false`), for example `"aero_ride_height": false` neglects the suspension moving the ride height, so the aero stays at the static ride height. The defaults are the thesis model. Part 7 shows what each switch is worth in lap time.

In [ ]:
car = lapsim.load_car()   # reads settings.json: tire, suspension, brakes, correlation, effects and car

## 3. Check the inputs

In [ ]:
lapsim.plot_tire(car)                       # Pacejka lateral force and aligning moment
plt.show()
display(lapsim.shift_table(car).round(1))   # gear matrix: speed and rpm before / after each shift

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for gear_index in range(len(car.gear_ratios)):
    ax[0].plot(car.v_grid * 3.6, car.gear_force_grid[gear_index], lw=0.8, label=f"gear {gear_index + 1}")
ax[0].plot(car.v_grid * 3.6, car.f_engine_grid, "k", lw=2, label="best gear")
ax[0].plot(car.v_grid * 3.6, [lapsim.resistance_force(car, speed_values) for speed_values in car.v_grid], "k--", label="drag + rolling")
ax[0].set(xlabel="speed [km/h]", ylabel="force at the wheels [N]", title="Tractive force"); ax[0].legend(fontsize=7)
speed_values = np.linspace(1, car.top_speed, 100)
ax[1].plot(speed_values * 3.6, [lapsim.tire_lateral_limit(car, speed) / car.mass / lapsim.G for speed in speed_values], label="max lateral")
ax[1].plot(speed_values * 3.6, [lapsim.tire_brake_limit(car, speed) / car.mass / lapsim.G for speed in speed_values], label="max braking")
ax[1].plot(speed_values * 3.6, [lapsim.tire_accel_limit(car, speed) / car.mass / lapsim.G for speed in speed_values], label="max tire accel")
ax[1].set(xlabel="speed [km/h]", ylabel="g", title="Tire limits vs. speed"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"Top speed: {car.top_speed * 3.6:.0f} km/h")

## 4. Single scenarios (thesis chapter 3)
Each scenario is simulated on its own. Weight transfer is on by default (`weight_transfer_on=False` switches it off).
`solver="distance"` (fast) or `solver="time"` (more accurate for short runs).

In [ ]:
acceleration_result = lapsim.run_acceleration(car, distance=75)                       # Formula Student acceleration
braking_result = lapsim.run_braking(car, initial_speed=100 / 3.6, speed=35 / 3.6)  # typical hairpin braking
skidpad_result = lapsim.run_cornering(car, radius=9.125, rotation=360)          # skidpad

print(f"Acceleration 75 m : {acceleration_result['time']:.3f} s, {acceleration_result['end_speed'] * 3.6:.1f} km/h at the end")
print(f"Braking 100 -> 35 : {braking_result['time']:.3f} s over {braking_result['distance']:.1f} m")
print(f"Skidpad lap       : {skidpad_result['time']:.3f} s at {skidpad_result['lat_accel']:.2f} g")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
trace = acceleration_result["trace"]
ax[0].plot(trace.s, trace.v_kmh); ax[0].set(xlabel="distance [m]", ylabel="speed [km/h]", title="acceleration")
ax[1].plot(trace.s, trace.ax / lapsim.G); ax[1].set(xlabel="distance [m]", ylabel="ax [g]", title="acceleration: grip limit, then power limit")
ax[2].plot(trace.s, trace["gear"]); ax[2].set(xlabel="distance [m]", ylabel="gear", title="gear")
plt.tight_layout(); plt.show()

lapsim.plot_ggv(car, speeds=(10, 20, 30))   # GG diagram at three speeds
plt.show()

## 5. Tracks
`tracks` already holds the autocross and the endurance track traced from the competition maps (about 0.7 m per pixel, so approximate).
The thesis lets you build a track from four kinds of data. Use whichever you have:

```python
# 1. radius & length table (radius positive = left, negative = right, 1e5 = straight)
mytrack = lapsim.track_from_radius_length(radius, section_length, closed=True)
mytrack = lapsim.load_radius_length_csv("my_track.csv")

# 2. GPS log (latitude, longitude in degrees)
mytrack = lapsim.track_from_gps("gps lap", lat, lon, closed=True, start_finish="auto")

# 3. x, y coordinates in metres
mytrack = lapsim.track_from_xy("xy lap", x, y, closed=True)

# 4. data logger: time [s], speed [km/h], lateral acceleration [g]
mytrack = lapsim.track_from_accel_speed("logged lap", t, speed_kmh, ay_g, filter="butterworth", cutoff_hz=2.0)

mytrack = lapsim.fine_mesh(mytrack)     # small steps in the corners (always do this)
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (name, trk) in zip(axes, tracks.items()):
    lapsim.plot_track(trk, trk.radius.clip(max=100), "corner radius [m]", ax=ax)
    ax.set_title(f"{name}: {trk.length:.0f} m, {len(trk.s)} points after the fine mesh")
plt.tight_layout(); plt.show()

## 6. Lap simulation (thesis chapter 4)
Forward / reverse solver: apexes, accelerate from every apex, brake to every apex, take the minimum, add the exact braking points.
`start="standing"` starts from 0 m/s; `start="flying"` (closed track) starts and ends at the same speed.

In [ ]:
autocross_lap = lapsim.simulate_lap(car, tracks["autocross"], start="standing")
endurance_lap = lapsim.simulate_lap(car, tracks["endurance"], start="flying")
print(f"Autocross : {autocross_lap['time']:.2f} s   Endurance lap (flying): {endurance_lap['time']:.2f} s")
display(pd.DataFrame({"autocross": lapsim.lap_kpis(car, autocross_lap), "endurance lap": lapsim.lap_kpis(car, endurance_lap)}).round(2))

In [ ]:
lapsim.plot_lap(car, autocross_lap)                       # speed, ax/ay, throttle/brake, gear/rpm, steering
plt.show()
for column in ("v_kmh", "TPS [%]", "steering [deg]", "pBrakeF [bar]"):
    lapsim.plot_lap_map(autocross_lap, column)             # track map coloured by any channel
    plt.show()

## 7. 2026 points, goals and what-if
The four dynamic events are simulated and scored with the FSAE rules (D.9 to D.12) against the 2026 Michigan field.
Static events (cost, presentation, design) use our real 2026 scores. Efficiency is set by hand because the lapsim does not model fuel use.

In [ ]:
events = lapsim.run_all_events(car, tracks)
table, summary = lapsim.points_2026(events, results, efficiency=0)
display(table.round(2))
print(f"Total with this car: {summary['total']:.1f} points  ->  would place {summary['place']} of {summary['teams']}")
print(f"2026 actual:         {summary['actual total']:.1f} points  ->  placed {summary['actual place']}")

Notes
- If the sim is faster than the field's best time, the event shows max points (the rules make you the new fastest time).
- Endurance: standing-start first lap, then flying laps for 22 km. Real times include traffic, mistakes and the driver change, so the sim is faster. Use the correlation factors (part 11).

In [ ]:
columns = {"acceleration": "BestTime", "skidpad": "BestTime", "autocross": "BestTime", "endurance": "AdjustedTime"}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, event in zip(axes, lapsim.SCORING):
    times = np.linspace(fastest_times[event], lapsim.SCORING[event][0] * fastest_times[event] * 1.02, 200)
    ax.plot(times, [lapsim.event_points(event, time_value, fastest_times[event]) for time_value in times], label="rules formula")
    field_times = pd.to_numeric(results[event][columns[event]], errors="coerce").dropna()
    ax.scatter(field_times, [lapsim.event_points(event, time_value, fastest_times[event]) for time_value in field_times], s=8, c="gray", label="2026 teams")
    ax.axvline(events[event]["time"], color="C3", label="our sim")
    ax.set(title=event, xlabel="time [s]", ylabel="points")
axes[0].legend(); plt.tight_layout(); plt.show()

### Goal setting: what time do we need for a target score?

In [ ]:
goals = {"acceleration": 90, "skidpad": 60, "autocross": 100, "endurance": 200}   # edit me (endurance includes 25 for finishing)

needed = lapsim.goal_times(goals, results)
needed["sim time [s]"] = [events[e]["time"] for e in needed.index]
needed["gap [s]"] = needed["sim time [s]"] - needed["needed time [s]"]   # negative = we already beat the goal
display(needed.round(3))

gain = {e: lapsim.points_per_percent(e, events[e]["time"], fastest_times[e]) for e in lapsim.SCORING}
display(pd.DataFrame({"points for a 1 % faster time": gain}).round(2))

### What-if: change anything and see times and points
Use dotted names for the sub-models, for example `"suspension.spring_front_lbs_in"`, `"tire.mux"`, `"correlation.aero"`.

In [ ]:
changes = {"mass": car.mass - 10, "cz_total": car.cz_total + 0.5}   # edit me
display(lapsim.compare_designs(car, changes, tracks, results).round(2))

### Which effects matter? Switch them off one at a time
Every switch in `effects` (part 2) is flipped on its own and the endurance flying lap is run again. A positive change means the lap gets slower. The aero switches do nothing while `aero_mode="constant"`, so this runs on the aero map car (part 8). To neglect several effects together, set them in part 2 or use `lapsim.with_params(car, **{"effects.aero_ride_height": False, ...})`.

In [ ]:
car_for_study = lapsim.with_params(car, aero_mode="map", aero_map=lapsim.AeroMap.placeholder())
flying_lap = lambda c: lapsim.simulate_lap(c, tracks["endurance"], "flying", channels=False)["time"]
display(lapsim.effects_study(car_for_study, flying_lap)[["now", "base", "result when flipped", "change", "change [%]"]])

## 8. Aero map (thesis chapters 6 and 7)
With `aero_mode="map"` the downforce, drag and aero balance depend on the car's **front and rear ride height, roll and yaw**.
The ride heights are found by iteration (downforce squeezes the springs, which changes the downforce) and also move with
weight transfer through the anti-dive / anti-squat percentages. **`AeroMap.placeholder()` is made up. Load your CFD data with
`lapsim.AeroMap.from_csv("my_map.csv")` (columns frh, rrh, roll, yaw, czt, ab, cx).**

In [ ]:
aero_map = lapsim.AeroMap.placeholder()
car_map = lapsim.with_params(car, aero_mode="map", aero_map=aero_map)

lapsim.plot_aero_map(aero_map); plt.show()                       # aero viewer
display(lapsim.aero_lookup(aero_map, frh=17, rrh=20, roll=0.8, yaw=6.5).round(3))   # thesis worked example
lapsim.plot_aero_envelope(car_map); plt.show()                   # CzT / AB vs speed: straight, roll, yaw

In [ ]:
# Static ride height decision: every front x rear combination, scored over a speed range of interest
sweep = lapsim.static_ride_height_sweep(car_map, [30, 35, 40, 45], [30, 35, 40, 45], speed_range_kmh=(20, 100))
display(sweep[sweep["ok (no bottoming)"]].sort_values("median CzT", ascending=False).head(6).round(2))

# The same lap with the aero map: ride heights and aero parameters now change around the lap
lap_map = lapsim.simulate_lap(car_map, tracks["autocross"], start="standing")
print(f"Autocross: {lap_map['time']:.2f} s with the aero map, {autocross_lap['time']:.2f} s with constant aero")
lapsim.plot_ride_height_envelope(lap_map); plt.show()
display(lapsim.lap_kpis(car_map, lap_map)[["Median CzT", "Median AB [%]", "Median Cx", "Min FRH [mm]", "Min RRH [mm]"]].round(2))

## 9. Suspension dynamics (thesis chapter 5)
Rates calculator, quarter car model, damping curves, parameter sweeps and the virtual 7-post rig (full car, 7 degrees of freedom).

In [ ]:
display(pd.Series(car.rates).round(3).rename("value").to_frame())      # rates calculator
display(pd.Series(lapsim.quarter_car(car)).round(2).rename("value").to_frame())   # natural frequencies, critical damping
display(lapsim.damping_curves(car).iloc[::10].round(0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
curves = lapsim.damping_curves(car)
curves.plot(ax=axes[0]); axes[0].set(title="Damping curves (force at the wheel)")
lapsim.quarter_car_response(car, "front").plot(ax=axes[1]); axes[1].set(title="Quarter car, 10 mm bump (front)")
for mode in ("heave", "roll", "pitch"):
    response = lapsim.seven_post_free_response(car, mode)
    axes[2].plot(response.index, response["tire load LF [N]"], label=mode)
axes[2].set(title="7-post free response: tire load LF [N]", xlabel="t [s]"); axes[2].legend()
plt.tight_layout(); plt.show()
display(pd.DataFrame({mode: lapsim.seven_post_summary(car, mode) for mode in ("heave", "roll", "pitch")}).round(0))

In [ ]:
# Parameter sweep: change one parameter over a range and watch the results (any dotted parameter name works)
sweep = lapsim.parameter_sweep(car, "suspension.spring_front_lbs_in", range(200, 550, 50), lapsim.rates_summary)
display(sweep[["heave_stiffness_front", "roll_stiffness_front", "mechanical_balance", "roll_gradient"]].round(2))
display(lapsim.parameter_sweep(car, "suspension.zeta_front_hs", [0.5, 0.75, 1.0, 1.25, 1.5], lapsim.seven_post_summary, mode="heave").round(0))

## 10. Yaw moment diagram (thesis chapter 8)
Steady-state cornering for many combinations of steer and body side slip angle, with the Pacejka tire, lateral weight transfer and aero.
**Balance** = yaw moment at the maximum lateral acceleration (positive = oversteer in left turns, near zero is good).
**Grip** = lateral acceleration. **Stability** and **control** = how the yaw moment changes with side slip and steer.

In [ ]:
yaw_moment_result = lapsim.yaw_moment_diagram(car, speed=20.0)           # constant speed [m/s]; use radius=9.125 for a constant radius diagram
display(pd.Series(yaw_moment_result["kpis"]).round(2).rename("value").to_frame())
lapsim.plot_yaw_moment_diagram(yaw_moment_result, lapsim.yaw_moment_diagram(car, speed=30.0), labels=["20 m/s", "30 m/s"])
plt.show()
display(lapsim.ymd_speed_sensitivity(car, [10, 15, 20, 25, 30]).round(1))
display(lapsim.parameter_sweep(car, "aero_balance_front", [35, 40, 45, 50, 55], lapsim.ymd_summary, speed=25.0).round(1))

## 11. Correlation (thesis 4.5)
The sim is always a bit too fast. Match it to the real car by adding correction factors one at a time:
aero, engine power, longitudinal grip, tire load sensitivity. The thesis values are used as the example. Set `target` to a real lap time.

In [ ]:
steps = [("Aero", {"aero": 0.5}),
         ("Engine power", {"engine_power": 0.8}),
         ("Longitudinal grip", {"mux_accel": 0.9, "mux_brake": 0.9}),
         ("Load sensitivity", {"load_sensitivity": 3})]
target = 110.0   # s, real flying lap time (edit me)
display(lapsim.correlation_study(car, tracks["endurance"], steps, start="flying", target_time=target).round(2))

# Or fit one factor so that a single scenario matches a measurement, as in the thesis:
factor = lapsim.fit_correlation(car, "mux_accel", 4.80, lambda candidate_car: lapsim.run_acceleration(candidate_car, distance=75)["time"])
print(f"mux_accel = {factor:.3f} gives a 75 m time of 4.80 s")

### Check against the thesis
The code reproduces the numbers printed in the thesis (baseline car):

In [ ]:
display(lapsim.thesis_checks().round(3))